# Lesson 0012 — Kalman filter from scratch

Implement a complete position–velocity Kalman filter using NumPy. You will write the three cells marked **TODO**; the remaining cells provide immediate checks and a visible simulation.

**Outcome:** a recursive estimator that predicts every second, corrects on every fourth GPS reading, and plots its own uncertainty.

Work from top to bottom. When a check fails, fix the preceding TODO cell before continuing. Ask your teacher about any formula, shape, or failure that remains unclear.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## Model

The state is `mean = [position, velocity]`. For one time step `dt`:

$$F = \begin{bmatrix}1 & \Delta t \\ 0 & 1\end{bmatrix}, \qquad \hat{x}^{-}=F\hat{x}^{+}$$

Unknown acceleration with standard deviation $\sigma_a$ contributes:

$$Q = \sigma_a^2 \begin{bmatrix}\Delta t^4/4 & \Delta t^3/2 \\ \Delta t^3/2 & \Delta t^2\end{bmatrix}$$

In [ ]:
def acceleration_process_covariance(
    dt: float,
    acceleration_std: float,
) -> np.ndarray:
    """Return the 2×2 process covariance caused by unknown acceleration."""
    # TODO: implement Q from the equation above.
    raise NotImplementedError

In [ ]:
expected_q = np.array([[0.01, 0.02], [0.02, 0.04]])
actual_q = acceleration_process_covariance(dt=1.0, acceleration_std=0.2)
np.testing.assert_allclose(actual_q, expected_q, atol=1e-12)
assert actual_q.shape == (2, 2)
print("✓ Process covariance is correct")

## Prediction

Move both parts of the belief:

$$\hat{x}^{-}=F\hat{x}^{+}, \qquad P^{-}=FP^{+}F^T+Q$$

In [ ]:
def predict(
    mean: np.ndarray,
    covariance: np.ndarray,
    dt: float,
    acceleration_std: float,
) -> tuple[np.ndarray, np.ndarray]:
    """Predict the position–velocity mean and covariance by one time step."""
    # TODO: construct F, call acceleration_process_covariance, and return x⁻ and P⁻.
    raise NotImplementedError

In [ ]:
initial_mean = np.array([0.0, 1.0])
initial_covariance = np.diag([0.04, 0.09])

predicted_mean, predicted_covariance = predict(
    initial_mean, initial_covariance, dt=1.0, acceleration_std=0.2
)

np.testing.assert_allclose(predicted_mean, [1.0, 1.0], atol=1e-12)
np.testing.assert_allclose(
    predicted_covariance,
    [[0.14, 0.11], [0.11, 0.13]],
    atol=1e-12,
)
np.testing.assert_allclose(predicted_covariance, predicted_covariance.T, atol=1e-12)
print("✓ Prediction matches the numerical example")

## Position measurement correction

GPS measures position only, so $H=[1\;0]$ and $R=\sigma_z^2$:

$$y=z-H\hat{x}^{-}$$
$$S=HP^{-}H^T+R$$
$$K=P^{-}H^TS^{-1}$$
$$\hat{x}^{+}=\hat{x}^{-}+Ky$$
$$P^{+}=(I-KH)P^{-}$$

In [ ]:
def correct_position(
    predicted_mean: np.ndarray,
    predicted_covariance: np.ndarray,
    measurement: float,
    measurement_std: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    """Correct a predicted belief with one scalar position measurement.

    Return corrected mean, corrected covariance, Kalman gain, and innovation.
    """
    # TODO: implement the five correction equations above.
    raise NotImplementedError

In [ ]:
corrected_mean, corrected_covariance, gain, innovation = correct_position(
    predicted_mean,
    predicted_covariance,
    measurement=1.2,
    measurement_std=0.5,
)

np.testing.assert_allclose(innovation, 0.2, atol=1e-12)
np.testing.assert_allclose(gain, [0.35897436, 0.28205128], atol=1e-8)
np.testing.assert_allclose(corrected_mean, [1.07179487, 1.05641026], atol=1e-8)
np.testing.assert_allclose(
    corrected_covariance,
    [[0.08974359, 0.07051282], [0.07051282, 0.09897436]],
    atol=1e-8,
)
assert corrected_covariance[0, 0] < predicted_covariance[0, 0]
print("✓ Measurement correction matches the complete Kalman cycle")

## Run the recursive filter

This cell supplies the experiment. It repeatedly calls your `predict` and `correct_position` functions. GPS arrives every fourth second.

In [ ]:
def simulate_filter(
    *,
    steps: int = 40,
    dt: float = 1.0,
    filter_acceleration_std: float = 0.18,
    gps_std: float = 0.5,
    gps_interval: int = 4,
    seed: int = 7,
) -> dict[str, np.ndarray]:
    rng = np.random.default_rng(seed)
    truth = np.array([0.0, 1.0])
    mean = np.array([0.0, 1.0])
    covariance = np.diag([0.04, 0.09])

    times = [0.0]
    truth_positions = [truth[0]]
    estimated_positions = [mean[0]]
    position_sigmas = [np.sqrt(covariance[0, 0])]
    gps_times = []
    gps_positions = []

    for step in range(1, steps + 1):
        actual_acceleration = 0.10 * np.sin(step / 5) + rng.normal(0.0, 0.05)
        truth = np.array([
            truth[0] + truth[1] * dt + 0.5 * actual_acceleration * dt**2,
            truth[1] + actual_acceleration * dt,
        ])

        mean, covariance = predict(
            mean, covariance, dt=dt, acceleration_std=filter_acceleration_std
        )

        if step % gps_interval == 0:
            measurement = truth[0] + rng.normal(0.0, gps_std)
            mean, covariance, _, _ = correct_position(
                mean, covariance, measurement, measurement_std=gps_std
            )
            gps_times.append(step * dt)
            gps_positions.append(measurement)

        times.append(step * dt)
        truth_positions.append(truth[0])
        estimated_positions.append(mean[0])
        position_sigmas.append(np.sqrt(covariance[0, 0]))

    return {
        "time": np.asarray(times),
        "truth_position": np.asarray(truth_positions),
        "estimated_position": np.asarray(estimated_positions),
        "position_sigma": np.asarray(position_sigmas),
        "gps_time": np.asarray(gps_times),
        "gps_position": np.asarray(gps_positions),
    }

In [ ]:
result = simulate_filter()
error = result["estimated_position"] - result["truth_position"]
rmse = np.sqrt(np.mean(error**2))
inside_two_sigma = np.mean(np.abs(error) <= 2 * result["position_sigma"])

fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(
    result["time"],
    result["estimated_position"] - 2 * result["position_sigma"],
    result["estimated_position"] + 2 * result["position_sigma"],
    color="tab:blue",
    alpha=0.14,
    label="estimate ±2σ",
)
ax.plot(result["time"], result["truth_position"], color="tab:green", label="truth")
ax.plot(result["time"], result["estimated_position"], color="tab:blue", label="Kalman estimate")
ax.scatter(result["gps_time"], result["gps_position"], color="tab:orange", label="GPS", zorder=3)
ax.set(xlabel="time, s", ylabel="position, m", title="Recursive position–velocity Kalman filter")
ax.grid(alpha=0.2)
ax.legend()
plt.show()

print(f"Position RMSE: {rmse:.3f} m")
print(f"Truth inside reported ±2σ: {inside_two_sigma:.1%} of steps")

## Experiments

After the default run works:

1. Run `simulate_filter(gps_std=1.2)`. Explain what changes in the corrections and uncertainty.
2. Run `simulate_filter(filter_acceleration_std=0.03)`. Check how often truth lies inside the reported `±2σ` interval. Explain why a smooth-looking estimate can still be overconfident.
3. Restore the defaults before saving the notebook.

## Short report — edit this cell

1. Why does uncertain acceleration create off-diagonal position–velocity covariance?
2. Why can a position-only GPS measurement correct velocity?
3. What observable behavior tells you that `Q` is too small?